In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch

# Import desde el paquete instalado por pip (git+https://github.com/facebookresearch/sam2.git)
from sam2.build_sam import build_sam2_video_predictor

# =========================
# CONFIG
# =========================
VIDEO_PATH = r"VideosAnalisis\clip 2 ‐ Hecho con Clipchamp.mp4"

# Checkpoint descargado por scripts/download_sam2_ckpt.py
SAM2_CKPT = os.path.join("checkpoints", "sam2.1_hiera_tiny.pt")

# El YAML de configs viene dentro del paquete/repo de SAM2.
SAM2_CFG = r"configs/sam2.1/sam2.1_hiera_t.yaml"

# Salidas
FRAMES_DIR = "./_sam2_frames"
OUT_DIR = "./outputs"
OUT_CSV = os.path.join(OUT_DIR, "ball_sam2_track.csv")
OUT_VIDEO = os.path.join(OUT_DIR, "ball_sam2_track.mp4")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# NUEVO: ajustes de UI/Frame inicial
# =========================
ROI_WINDOW_NAME = "Selecciona la pelota (ROI)"
ROI_PREVIEW_MAX_W = 960     # ancho máximo de la previsualización
ROI_PREVIEW_MAX_H = 540     # alto máximo de la previsualización

INIT_FRAME_GUESS = 10       # en vez de 0 (cámbialo si quieres)
NONBLACK_SEARCH_LIMIT = 200 # cuántos frames máximo buscar para evitar negro


def ensure_dirs():
    os.makedirs(FRAMES_DIR, exist_ok=True)
    os.makedirs(OUT_DIR, exist_ok=True)


def extract_frames(video_path, frames_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir el video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    for i in range(frame_count):
        ret, frame = cap.read()
        if not ret:
            frame_count = i
            break
        cv2.imwrite(os.path.join(frames_path, f"{i:06d}.jpg"), frame)

    cap.release()
    return fps, W, H, frame_count


def read_frame_from_video(video_path, frame_idx):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None


def is_black_frame(frame, thr_mean=8.0):
    # Umbral simple: media de luminancia muy baja => frame negro
    if frame is None:
        return True
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return float(gray.mean()) < thr_mean


def pick_nonblack_frame_idx(video_path, start_idx, frame_count, search_limit=200):
    start_idx = max(0, min(int(start_idx), max(0, frame_count - 1)))
    max_idx = min(frame_count - 1, start_idx + int(search_limit))

    for idx in range(start_idx, max_idx + 1):
        fr = read_frame_from_video(video_path, idx)
        if fr is not None and not is_black_frame(fr):
            return idx, fr

    # Si no encontró, cae al start_idx aunque sea negro (para no petar)
    fallback = read_frame_from_video(video_path, start_idx)
    return start_idx, fallback


def load_or_create_frame_jpg(frames_path, video_path, frame_idx):
    """
    Devuelve el frame BGR para frame_idx.
    - Si existe el JPG, lo carga.
    - Si no existe, lo lee del vídeo y lo guarda como JPG.
    """
    frame_file = os.path.join(frames_path, f"{int(frame_idx):06d}.jpg")
    if os.path.exists(frame_file):
        fr = cv2.imread(frame_file)
        if fr is not None:
            return fr

    fr = read_frame_from_video(video_path, frame_idx)
    if fr is None:
        return None
    cv2.imwrite(frame_file, fr)
    return fr


def resize_for_preview(frame, max_w=960, max_h=540):
    """
    Redimensiona SOLO para previsualización manteniendo aspect ratio.
    Devuelve: frame_preview, scale_x, scale_y (para mapear ROI->original)
    """
    H, W = frame.shape[:2]
    scale = min(max_w / W, max_h / H, 1.0)  # nunca agrandar
    new_w = int(round(W * scale))
    new_h = int(round(H * scale))
    preview = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA) if scale < 1.0 else frame.copy()
    scale_x = W / new_w
    scale_y = H / new_h
    return preview, scale_x, scale_y


def select_roi_small_window(frame_bgr):
    """
    Muestra una ventana redimensionable y pequeña. Selección ROI se hace
    sobre el frame reescalado, y se devuelve la ROI reescalada al original.
    """
    preview, sx, sy = resize_for_preview(frame_bgr, ROI_PREVIEW_MAX_W, ROI_PREVIEW_MAX_H)

    # Ventana redimensionable + tamaño inicial controlado
    cv2.namedWindow(ROI_WINDOW_NAME, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(ROI_WINDOW_NAME, preview.shape[1], preview.shape[0])

    print("[INFO] Selecciona la PELOTA con una caja y pulsa ENTER. (ESC para cancelar)")
    roi = cv2.selectROI(ROI_WINDOW_NAME, preview, fromCenter=False, showCrosshair=True)
    cv2.destroyWindow(ROI_WINDOW_NAME)

    x, y, w, h = roi
    if w == 0 or h == 0:
        raise RuntimeError("ROI vacío. Vuelve a ejecutar y selecciona una caja válida.")

    # Escalar ROI de preview -> original
    x0 = float(x) * sx
    y0 = float(y) * sy
    x1 = float(x + w) * sx
    y1 = float(y + h) * sy

    box = np.array([x0, y0, x1, y1], dtype=np.float32)  # xyxy en coords del frame original
    return box


def main():
    ensure_dirs()

    if not os.path.exists(SAM2_CKPT):
        raise FileNotFoundError(
            f"No existe el checkpoint: {SAM2_CKPT}\n"
            f"Ejecuta antes: python scripts/download_sam2_ckpt.py"
        )

    # 1) extraer frames
    video_stem = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
    frames_path = os.path.join(FRAMES_DIR, video_stem)
    os.makedirs(frames_path, exist_ok=True)

    existing_jpgs = [f for f in os.listdir(frames_path) if f.lower().endswith(".jpg")]

    if len(existing_jpgs) == 0:
        print(f"[INFO] Extrayendo frames a: {frames_path}")
        fps, W, H, frame_count = extract_frames(VIDEO_PATH, frames_path)
    else:
        cap = cv2.VideoCapture(VIDEO_PATH)
        if not cap.isOpened():
            raise RuntimeError(f"No se pudo abrir el video: {VIDEO_PATH}")
        fps = cap.get(cv2.CAP_PROP_FPS)
        W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()

    # 2) elegir frame inicial NO NEGRO (no usar el primero)
    init_frame_idx, init_frame = pick_nonblack_frame_idx(
        VIDEO_PATH,
        start_idx=INIT_FRAME_GUESS,
        frame_count=frame_count,
        search_limit=NONBLACK_SEARCH_LIMIT
    )
    if init_frame is None:
        raise RuntimeError("No pude leer un frame inicial del vídeo.")

    # Asegura que ese frame exista como JPG en frames_path (SAM2 usa carpeta)
    init_frame = load_or_create_frame_jpg(frames_path, VIDEO_PATH, init_frame_idx)
    if init_frame is None:
        raise RuntimeError(f"No pude cargar/crear el frame {init_frame_idx} para ROI.")

    print(f"[INFO] Usando frame inicial: {init_frame_idx:06d}.jpg (evitando posibles frames negros).")

    # 3) seleccionar ROI en init_frame_idx con ventana pequeña/redimensionable
    box = select_roi_small_window(init_frame)

    # 4) SAM2 predictor
    print(f"[INFO] Cargando SAM2 en {DEVICE} ...")
    predictor = build_sam2_video_predictor(SAM2_CFG, SAM2_CKPT, device=DEVICE)

    # init_state con carpeta de frames
    with torch.inference_mode():
        try:
            state = predictor.init_state(video_path=frames_path)
        except TypeError:
            state = predictor.init_state(frames_path)

    OBJ_ID = 1
    with torch.inference_mode():
        try:
            predictor.add_new_points_or_box(state, frame_idx=int(init_frame_idx), obj_id=OBJ_ID, box=box)
        except TypeError:
            predictor.add_new_points_or_box(state, frame_idx=int(init_frame_idx), object_id=OBJ_ID, box=box)

    # 5) propagar
    print("[INFO] Propagando máscara en el vídeo...")
    rows = []
    masks_by_frame = {}

    with torch.inference_mode():
        for f_idx, obj_ids, masks in predictor.propagate_in_video(state):
            f_idx = int(f_idx)

            obj_ids_list = [int(o) for o in obj_ids]
            if OBJ_ID not in obj_ids_list:
                rows.append({"frame": f_idx, "x": np.nan, "y": np.nan, "area": 0})
                continue

            j = obj_ids_list.index(OBJ_ID)
            mask = masks[j]

            mask_np = mask.squeeze().detach().float().cpu().numpy()
            binmask = (mask_np > 0).astype(np.uint8)

            area = int(binmask.sum())
            if area < 5:
                rows.append({"frame": f_idx, "x": np.nan, "y": np.nan, "area": area})
                continue

            ys, xs = np.where(binmask > 0)
            cx = float(xs.mean())
            cy = float(ys.mean())

            rows.append({"frame": f_idx, "x": cx, "y": cy, "area": area})
            masks_by_frame[f_idx] = binmask

    df = pd.DataFrame(rows).sort_values("frame")
    df.to_csv(OUT_CSV, index=False)
    print(f"[OK] CSV guardado: {OUT_CSV}")

    # 6) render overlay
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(OUT_VIDEO, fourcc, fps, (W, H))

    print("[INFO] Generando vídeo con overlay...")
    for f_idx in range(frame_count):
        frame_file = os.path.join(frames_path, f"{f_idx:06d}.jpg")
        frame = cv2.imread(frame_file)

        # Si faltan JPGs (porque no reextraíste o el conteo difiere), intentamos leer del vídeo
        if frame is None:
            frame = read_frame_from_video(VIDEO_PATH, f_idx)
            if frame is None:
                break

        if f_idx in masks_by_frame:
            bm = masks_by_frame[f_idx]
            overlay = frame.copy()
            overlay[bm.astype(bool)] = (0, 255, 0)
            frame = cv2.addWeighted(frame, 0.75, overlay, 0.25, 0)

            row = df[df["frame"] == f_idx]
            if len(row) == 1 and not np.isnan(row.iloc[0]["x"]):
                cx, cy = int(row.iloc[0]["x"]), int(row.iloc[0]["y"])
                cv2.circle(frame, (cx, cy), 6, (0, 255, 255), -1)
                cv2.putText(frame, "BALL(SAM2)", (cx + 10, cy - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        writer.write(frame)

    writer.release()
    print(f"[OK] Vídeo guardado: {OUT_VIDEO}")


if __name__ == "__main__":
    main()


[INFO] Extrayendo frames a: ./_sam2_frames\clip 2 ‐ Hecho con Clipchamp
[INFO] Selecciona la PELOTA con una caja y pulsa ENTER. (ESC para cancelar)


RuntimeError: ROI vacío. Vuelve a ejecutar y selecciona una caja válida.